In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDaC/notebooks


## Env

In [2]:
from src.utils.env import prepare_environment

_ = prepare_environment("../api_keys")

## Model

In [ ]:
from src.configs import art_configs
from src.configs import vllm_configs
from src.models import PathConfig

print("Available ART model configurations:")
print(art_configs.available_configs())

print("Available vLLM model configurations:")
print(vllm_configs.available_configs())

base_model = "unsloth/Qwen2.5-14B-Instruct"
project_name = "easy2hard_dac"

path_config = PathConfig(
    base_model=base_model,
    project_name=project_name,
)

Available ART model configurations:
['unsloth/Qwen2-7B', 'Qwen/Qwen2.5-32B-Instruct', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen3-14B', 'unsloth/Qwen3-14B-bnb-4bit', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Llama-4-Scout-17B-16E-Instruct']
Available vLLM model configurations:
['unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen3-14B-bnb-4bit', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit']


In [ ]:
from src.models import load_art_model

model = await load_art_model(path_config=path_config, print_full=True)

## Data

In [ ]:
from datasets import Dataset, load_dataset, DatasetDict

dataset_dict: DatasetDict = load_dataset(
    path="furonghuang-lab/Easy2Hard-Bench",
    name="E2H-AMC",
    split=None,
)  # type: ignore

train_data: Dataset = dataset_dict["train"]
test_data: Dataset = dataset_dict["eval"]

## Inference Clients

In [ ]:
from src.vllm_client import VllmClient, ArtClient, VllmRouter

inference_clients: list[VllmClient] = [ArtClient.from_art_model(model)]
vllm_server_ports = [8200]
for port in vllm_server_ports:
    inference_clients.append(
        VllmClient.from_connection(
            port=port,
            base_model=model.base_model,
            model_name=model.get_inference_name(),
        )
    )

vllm_router: VllmRouter = VllmRouter(inference_clients)

## Training and Inference 

In [ ]:
from experiments.easy2hard.trainer import Easy2HardTrainer
from src.trainer import PromptConfig, StopCriteria
import art

prompt_config = PromptConfig(
    system_root="dac_sys_prompt_gilad_root",
    system_inter="dac_sys_prompt_gilad_inter",
    system_leaf="dac_sys_prompt_gilad_leaf",
    tasks_depleted="tasks_depleted",
)

stop_criteria = StopCriteria(
    max_depth=1,
    max_tasks=5,
    max_rounds=5,
)

trainer = Easy2HardTrainer(
    model=model,
    vllm_router=vllm_router,
    path_config=path_config,
    prompt_config=prompt_config,
    stop_criteria=stop_criteria,
)

NameError: name 'model' is not defined

### Inference Test

In [ ]:
import random

idx = random.randint(0, len(train_data) - 1)
# idx = 228
print(f"Selected random index: {idx}")
sample = train_data[idx]
problem = sample["problem"]
true_answer = sample["answer"]

print(f"Problem: {problem}")
print(f"Answer: {true_answer}")
print("-" * 200)
print()

await trainer.sync_lora()
preds = await trainer.predict([sample], verbose=True, seed=0)

### Train

In [ ]:
from src.trainer import TrainingConfig

train_config = TrainingConfig(
    epochs=100,
    num_groups=5,
    group_size=50,
    train_log_steps=1,
    eval_log_steps=2,
    eval_size=250,
    verbose=False,
    min_reward_stdev=None,
    art_config=art.types.TrainConfig(learning_rate=1e-5),
)

try:
    await trainer.train(
        config=train_config,
        train_dataset=train_data.to_list(),
        eval_dataset=test_data.to_list(),
    )
finally:
    trainer.close()